# AI + CAD Optimization of Soft Robotic Gripper Hub
Complete notebook for loading the final dataset, plotting results, ranking designs, and running ML analysis.

## Step 0: Install package for overlap-safe labels
Run this once in Colab or Jupyter if `adjustText` is not installed.

In [ ]:
!pip install adjustText

## Step 1: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score
from adjustText import adjust_text


## Step 2: Load dataset

In [ ]:
df = pd.read_csv("gripper_dataset_final.csv")
print(df.head())
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())


## Step 3: Basic checks

In [ ]:
print(df.info())
print(df.describe())


## Step 4: Add derived comparison columns against V1

In [ ]:
baseline_mass = df.loc[df["version"] == "V1", "mass"].iloc[0]
baseline_stress = df.loc[df["version"] == "V1", "max_stress"].iloc[0]
baseline_sf = df.loc[df["version"] == "V1", "safety_factor"].iloc[0]

df["mass_reduction_pct_vs_V1"] = (baseline_mass - df["mass"]) / baseline_mass * 100
df["stress_change_pct_vs_V1"] = (df["max_stress"] - baseline_stress) / baseline_stress * 100
df["sf_change_pct_vs_V1"] = (df["safety_factor"] - baseline_sf) / baseline_sf * 100

df[[
    "version",
    "mass",
    "mass_reduction_pct_vs_V1",
    "max_stress",
    "stress_change_pct_vs_V1",
    "safety_factor",
    "sf_change_pct_vs_V1"
]].round(3)


## Step 5: Helper functions

In [ ]:
def minmax(series):
    smin = series.min()
    smax = series.max()
    if smax == smin:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - smin) / (smax - smin)

def labeled_scatter(df, x_col, y_col, title, xlabel, ylabel):
    plt.figure(figsize=(7, 5))
    plt.scatter(df[x_col], df[y_col])
    texts = []
    for _, row in df.iterrows():
        texts.append(plt.text(row[x_col], row[y_col], row["version"], fontsize=8))
    adjust_text(texts, arrowprops=dict(arrowstyle="->", lw=0.5))
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.show()

def actual_vs_predicted_plot(actual, predicted, labels, title, xlabel, ylabel):
    plt.figure(figsize=(7, 5))
    plt.scatter(actual, predicted)
    texts = []
    for a, p, lab in zip(actual, predicted, labels):
        texts.append(plt.text(a, p, lab, fontsize=8))
    adjust_text(texts, arrowprops=dict(arrowstyle="->", lw=0.5))
    min_val = min(float(np.min(actual)), float(np.min(predicted)))
    max_val = max(float(np.max(actual)), float(np.max(predicted)))
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", linewidth=1)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.show()


## Step 6: Main engineering plots

In [ ]:
labeled_scatter(df, "mass", "max_stress", "Mass vs Max Stress", "Mass (g)", "Max Stress (MPa)")
labeled_scatter(df, "mass", "safety_factor", "Mass vs Safety Factor", "Mass (g)", "Safety Factor")
labeled_scatter(df, "pocket_depth", "mass", "Pocket Depth vs Mass", "Pocket Depth (mm)", "Mass (g)")
labeled_scatter(df, "rib_thickness", "max_stress", "Rib Thickness vs Max Stress", "Rib Thickness (mm)", "Max Stress (MPa)")
labeled_scatter(df, "cutout_length", "safety_factor", "Cutout Length vs Safety Factor", "Cutout Length (mm)", "Safety Factor")


## Step 7: Rank designs using weighted engineering score

In [ ]:
df["score"] = (
    0.35 * (1 - minmax(df["mass"])) +
    0.25 * (1 - minmax(df["max_stress"])) +
    0.15 * (1 - minmax(df["max_disp"])) +
    0.25 * minmax(df["safety_factor"])
)

df["rank"] = df["score"].rank(ascending=False, method="dense").astype(int)
ranked_df = df.sort_values(["rank", "score"], ascending=[True, False]).reset_index(drop=True)

ranked_df[["rank", "version", "mass", "max_stress", "max_disp", "safety_factor", "score"]].round(4)


## Step 8: Machine learning setup

In [ ]:
feature_cols = [
    "pocket_depth",
    "through_cut",
    "rib_thickness",
    "cutout_length",
    "cutout_angle"
]

X = df[feature_cols]
loo = LeaveOneOut()


## Step 9: Predict `max_stress`

In [ ]:
y_stress = df["max_stress"]

lin_cv_pred_stress = cross_val_predict(LinearRegression(), X, y_stress, cv=loo)
rf_cv_pred_stress = cross_val_predict(
    RandomForestRegressor(n_estimators=300, random_state=42),
    X,
    y_stress,
    cv=loo
)

print("Linear Regression MAE:", mean_absolute_error(y_stress, lin_cv_pred_stress))
print("Linear Regression R2:", r2_score(y_stress, lin_cv_pred_stress))
print("Random Forest MAE:", mean_absolute_error(y_stress, rf_cv_pred_stress))
print("Random Forest R2:", r2_score(y_stress, rf_cv_pred_stress))

pd.DataFrame({
    "version": df["version"],
    "actual_max_stress": y_stress,
    "linear_cv_pred_max_stress": lin_cv_pred_stress,
    "rf_cv_pred_max_stress": rf_cv_pred_stress
}).round(4)


## Step 10: Actual vs predicted plots for `max_stress`

In [ ]:
actual_vs_predicted_plot(
    y_stress,
    lin_cv_pred_stress,
    df["version"],
    "Linear Regression: Actual vs Predicted Max Stress",
    "Actual Max Stress",
    "Predicted Max Stress"
)

actual_vs_predicted_plot(
    y_stress,
    rf_cv_pred_stress,
    df["version"],
    "Random Forest: Actual vs Predicted Max Stress",
    "Actual Max Stress",
    "Predicted Max Stress"
)


## Step 11: Predict `safety_factor`

In [ ]:
y_sf = df["safety_factor"]

lin_cv_pred_sf = cross_val_predict(LinearRegression(), X, y_sf, cv=loo)
rf_cv_pred_sf = cross_val_predict(
    RandomForestRegressor(n_estimators=300, random_state=42),
    X,
    y_sf,
    cv=loo
)

print("Linear Regression MAE:", mean_absolute_error(y_sf, lin_cv_pred_sf))
print("Linear Regression R2:", r2_score(y_sf, lin_cv_pred_sf))
print("Random Forest MAE:", mean_absolute_error(y_sf, rf_cv_pred_sf))
print("Random Forest R2:", r2_score(y_sf, rf_cv_pred_sf))

pd.DataFrame({
    "version": df["version"],
    "actual_safety_factor": y_sf,
    "linear_cv_pred_safety_factor": lin_cv_pred_sf,
    "rf_cv_pred_safety_factor": rf_cv_pred_sf
}).round(4)


## Step 12: Actual vs predicted plots for `safety_factor`

In [ ]:
actual_vs_predicted_plot(
    y_sf,
    lin_cv_pred_sf,
    df["version"],
    "Linear Regression: Actual vs Predicted Safety Factor",
    "Actual Safety Factor",
    "Predicted Safety Factor"
)

actual_vs_predicted_plot(
    y_sf,
    rf_cv_pred_sf,
    df["version"],
    "Random Forest: Actual vs Predicted Safety Factor",
    "Actual Safety Factor",
    "Predicted Safety Factor"
)


## Step 13: Feature importance for `max_stress`

In [ ]:
rf_stress = RandomForestRegressor(n_estimators=300, random_state=42)
rf_stress.fit(X, y_stress)

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_stress.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df)

plt.figure(figsize=(7, 5))
plt.bar(importance_df["feature"], importance_df["importance"])
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.title("Random Forest Feature Importance for Max Stress")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## Step 14: Final V1 vs V11 comparison

In [ ]:
df[df["version"].isin(["V1", "V11"])][
    ["version", "mass", "max_stress", "max_disp", "safety_factor"]
].round(4)
